# Comparing Distributions Across Multiple Columns

**Milestone: Multi-Column Distribution Comparison**  
**Project: Trivin Insight Engine**  
**Date: March 2026**

---

## Learning Objectives

By completing this notebook, you will be able to:

1. ✅ Understand what a distribution represents for a numeric column
2. ✅ Compare means and medians across multiple columns
3. ✅ Compare spread using range and standard deviation
4. ✅ Identify higher-variability and lower-variability columns
5. ✅ Use summary statistics to guide deeper exploratory analysis

---

## Why This Matters

Common beginner issues include:

- Analyzing columns in isolation
- Comparing raw values instead of distributions
- Ignoring spread and variability
- Drawing conclusions without enough context

Most useful EDA insights come from comparing variables side by side.

In this notebook, you will compare employee survey metrics across multiple columns using **summary statistics only**. No visualization or modeling is required.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

project_root = Path.cwd().resolve().parent
csv_path = project_root / "data" / "raw" / "employee_survey_2026_Q1.csv"

df = pd.read_csv(csv_path)

print(f"Dataset loaded from: {csv_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

Dataset loaded from: C:\Users\USER\OneDrive\Desktop\Data-science\S86_Trivin_Insight_Engine_Sprint3\data\raw\employee_survey_2026_Q1.csv
Shape: 30 rows x 9 columns


## Part 1: Inspect the DataFrame and Select Numeric Columns

Before comparing distributions, confirm what is in the dataset.

You should inspect:
- Shape
- Column names
- Data types
- A small sample of rows

Then select numeric columns that represent **measurable survey variables**. We will exclude `employee_id` because it is an identifier, not a distribution we want to analyze.

In [2]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

numeric_columns = [
    column
    for column in df.select_dtypes(include="number").columns
    if column != "employee_id"
]

print("\nNumeric columns selected for comparison:")
print(numeric_columns)

Column names:
['employee_id', 'department', 'survey_date', 'satisfaction_score', 'work_life_balance', 'management_support', 'career_growth', 'team_collaboration', 'response_text']

Data types:
employee_id            int64
department            object
survey_date           object
satisfaction_score     int64
work_life_balance      int64
management_support     int64
career_growth          int64
team_collaboration     int64
response_text         object
dtype: object

First 5 rows:


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
0,1001,Engineering,2026-01-15,7,8,6,5,9,Great team environment but limited growth oppo...
1,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
2,1003,Sales,2026-01-17,4,3,4,3,5,High pressure environment with poor work-life ...
3,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
4,1005,HR,2026-01-19,6,7,5,4,6,Decent work environment but lack career advanc...



Numeric columns selected for comparison:
['satisfaction_score', 'work_life_balance', 'management_support', 'career_growth', 'team_collaboration']


## Part 2: Compute Summary Statistics for Multiple Columns

A distribution comparison starts with a statistical summary across all selected numeric columns.

`describe()` gives you:
- Count
- Mean
- Standard deviation
- Minimum
- Quartiles
- Maximum

This creates a fast side-by-side view of the distributions.

In [3]:
summary_statistics = df[numeric_columns].describe().T
summary_statistics

,count,mean,std,min,25%,50%,75%,max
satisfaction_score,30.0,6.466667,1.960530,2.0,5.00,6.5,8.00,10.0
work_life_balance,30.0,6.300000,2.246069,1.0,5.00,7.0,8.00,10.0
management_support,30.0,5.966667,1.865907,2.0,5.00,6.0,7.75,9.0
career_growth,30.0,5.333333,1.917853,1.0,4.00,5.0,7.00,9.0
team_collaboration,30.0,6.833333,1.966676,3.0,5.25,7.0,8.00,10.0


## Part 3: Compare Means and Medians Across Columns

Central tendency tells you where values tend to cluster.

When comparing columns:
- **Mean** helps compare average level
- **Median** helps compare the middle value
- **Mean vs median** gives a clue about possible skew

A higher mean does **not** automatically mean a column is "better." It only tells you that values tend to be higher on that scale.

In [4]:
central_tendency = pd.DataFrame({
    "mean": df[numeric_columns].mean(),
    "median": df[numeric_columns].median(),
}).sort_values("mean", ascending=False)

central_tendency.round(2)

,mean,median
team_collaboration,6.83,7.0
satisfaction_score,6.47,6.5
work_life_balance,6.30,7.0
management_support,5.97,6.0
career_growth,5.33,5.0


## Part 4: Compare Range and Standard Deviation Across Columns

Spread tells you how tightly or widely values are distributed.

We will compare:
- **Range** = maximum - minimum
- **Standard deviation** = typical spread around the mean

Columns with larger spread are usually less consistent. Columns with smaller spread are usually more stable.

In [5]:
spread_comparison = pd.DataFrame({
    "min": df[numeric_columns].min(),
    "max": df[numeric_columns].max(),
    "range": df[numeric_columns].max() - df[numeric_columns].min(),
    "std_dev": df[numeric_columns].std(),
}).sort_values("std_dev", ascending=False)

spread_comparison.round(2)

,min,max,range,std_dev
work_life_balance,1,10,9,2.25
team_collaboration,3,10,7,1.97
satisfaction_score,2,10,8,1.96
career_growth,1,9,8,1.92
management_support,2,9,7,1.87


## Part 5: Build One Comparison Table for Central Tendency and Spread

A single comparison table makes multi-column reasoning much easier.

This table will include:
- Mean
- Median
- Minimum
- Maximum
- Range
- Standard deviation

Once everything is in one place, differences across columns become much easier to interpret.

In [6]:
comparison_table = pd.DataFrame({
    "mean": df[numeric_columns].mean(),
    "median": df[numeric_columns].median(),
    "min": df[numeric_columns].min(),
    "max": df[numeric_columns].max(),
    "range": df[numeric_columns].max() - df[numeric_columns].min(),
    "std_dev": df[numeric_columns].std(),
})

comparison_table = comparison_table.round(2)
comparison_table

,mean,median,min,max,range,std_dev
satisfaction_score,6.47,6.5,2,10,8,1.96
work_life_balance,6.30,7.0,1,10,9,2.25
management_support,5.97,6.0,2,9,7,1.87
career_growth,5.33,5.0,1,9,8,1.92
team_collaboration,6.83,7.0,3,10,7,1.97


## Part 6: Check Mean vs Median Differences to Reason About Skew

If the mean and median are close, the distribution may be fairly balanced.

If the mean is noticeably:
- **Higher than the median**, the column may be right-skewed
- **Lower than the median**, the column may be left-skewed

This is only a **conceptual clue**, not proof. It helps you decide which columns may need deeper investigation later.

In [7]:
comparison_table["mean_minus_median"] = (
    comparison_table["mean"] - comparison_table["median"]
).round(2)

comparison_table["possible_skew"] = comparison_table["mean_minus_median"].apply(
    lambda value: "possible right skew" if value > 0.25
    else "possible left skew" if value < -0.25
    else "roughly balanced"
)

comparison_table.sort_values("mean_minus_median", ascending=False)

,mean,median,min,max,range,std_dev,mean_minus_median,possible_skew
career_growth,5.33,5.0,1,9,8,1.92,0.33,possible right skew
satisfaction_score,6.47,6.5,2,10,8,1.96,-0.03,roughly balanced
management_support,5.97,6.0,2,9,7,1.87,-0.03,roughly balanced
team_collaboration,6.83,7.0,3,10,7,1.97,-0.17,roughly balanced
work_life_balance,6.30,7.0,1,10,9,2.25,-0.70,possible left skew


## Part 7: Identify Columns with Higher Variability

Now sort the comparison table to identify which variables are more volatile and which are more stable.

This helps answer questions such as:
- Which metric changes the most from employee to employee?
- Which metric appears more consistent?
- Which columns might deserve extra attention in deeper analysis?

In [8]:
variability_rank = comparison_table.sort_values(
    ["std_dev", "range"], ascending=False
)

variability_rank

,mean,median,min,max,range,std_dev,mean_minus_median,possible_skew
work_life_balance,6.30,7.0,1,10,9,2.25,-0.70,possible left skew
team_collaboration,6.83,7.0,3,10,7,1.97,-0.17,roughly balanced
satisfaction_score,6.47,6.5,2,10,8,1.96,-0.03,roughly balanced
career_growth,5.33,5.0,1,9,8,1.92,0.33,possible right skew
management_support,5.97,6.0,2,9,7,1.87,-0.03,roughly balanced


## Part 8: Flag Unusually Wide or Narrow Distributions Conceptually

A wide distribution is not automatically a problem, and a narrow distribution is not automatically ideal.

These comparisons simply help you notice where values are more spread out or more concentrated. That raises better follow-up questions for later EDA.

For example:
- A wide spread may suggest mixed employee experiences
- A narrow spread may suggest more consistency
- A column with lower average and high variability may deserve closer investigation

In [9]:
wide_distribution = variability_rank.index[0]
narrow_distribution = comparison_table.sort_values(["std_dev", "range"]).index[0]

print(f"Widest distribution by standard deviation: {wide_distribution}")
print(f"Narrowest distribution by standard deviation: {narrow_distribution}")

comparison_table.loc[[wide_distribution, narrow_distribution]]

Widest distribution by standard deviation: work_life_balance
Narrowest distribution by standard deviation: management_support


,mean,median,min,max,range,std_dev,mean_minus_median,possible_skew
work_life_balance,6.30,7.0,1,10,9,2.25,-0.70,possible left skew
management_support,5.97,6.0,2,9,7,1.87,-0.03,roughly balanced


## Part 9: Write Comparison-Based EDA Findings

Use the computed statistics to write plain-language findings.

This is the important mindset:
- Use numbers to compare columns
- Interpret differences carefully
- Raise questions instead of making final claims

The code below turns the comparison table into short EDA statements based on the actual dataset.

In [10]:
highest_mean = comparison_table["mean"].idxmax()
lowest_mean = comparison_table["mean"].idxmin()
highest_std = comparison_table["std_dev"].idxmax()
lowest_std = comparison_table["std_dev"].idxmin()
most_positive_skew_signal = comparison_table["mean_minus_median"].idxmax()
most_negative_skew_signal = comparison_table["mean_minus_median"].idxmin()

print("EDA findings based on distribution comparison:\n")
print(
    f"1. {highest_mean} has the highest average score "
    f"({comparison_table.loc[highest_mean, 'mean']:.2f})."
)
print(
    f"2. {lowest_mean} has the lowest average score "
    f"({comparison_table.loc[lowest_mean, 'mean']:.2f}), which may deserve extra attention."
)
print(
    f"3. {highest_std} shows the highest variability "
    f"(std = {comparison_table.loc[highest_std, 'std_dev']:.2f}), so employee responses are less consistent there."
)
print(
    f"4. {lowest_std} is the most stable metric "
    f"(std = {comparison_table.loc[lowest_std, 'std_dev']:.2f})."
)
print(
    f"5. {most_positive_skew_signal} has the strongest positive mean-minus-median signal "
    f"({comparison_table.loc[most_positive_skew_signal, 'mean_minus_median']:.2f}), suggesting possible right skew."
)
print(
    f"6. {most_negative_skew_signal} has the strongest negative mean-minus-median signal "
    f"({comparison_table.loc[most_negative_skew_signal, 'mean_minus_median']:.2f}), suggesting possible left skew."
)
print("\nThese comparisons guide deeper analysis, but they are not final conclusions on their own.")

EDA findings based on distribution comparison:

1. team_collaboration has the highest average score (6.83).
2. career_growth has the lowest average score (5.33), which may deserve extra attention.
3. work_life_balance shows the highest variability (std = 2.25), so employee responses are less consistent there.
4. management_support is the most stable metric (std = 1.87).
5. career_growth has the strongest positive mean-minus-median signal (0.33), suggesting possible right skew.
6. work_life_balance has the strongest negative mean-minus-median signal (-0.70), suggesting possible left skew.

These comparisons guide deeper analysis, but they are not final conclusions on their own.


## Practice Exercises

### Exercise 1
Create a new table that compares only these three columns:
- `satisfaction_score`
- `work_life_balance`
- `career_growth`

Then answer:
- Which column has the highest mean?
- Which has the widest range?
- Which appears most stable?

### Exercise 2
Sort the comparison table by `mean_minus_median`.

Then answer:
- Which column appears most right-skewed?
- Which appears most left-skewed?
- Which columns look closest to balanced?

### Exercise 3
Write three plain-language EDA findings using both central tendency and spread.

Avoid statements like "this column is best." Focus on distribution patterns instead.